

#1 - IMPORTAÇÃO


Primeiramente, importei a biblioteca Pandas (a mais apropriada para análise de dados).

In [1]:
#Importa a biblioteca pandas como pd
import pandas as pd

Em seguida, importei a base de dados como um dataframe.

In [2]:
#Importa na variável df o conteúdo do dataset dirty_cafe_sales.csv, cujo arquivo está no Google Drive
df = pd.read_csv('/content/drive/MyDrive/Trabalho/Google Colabs/Café Sales/Data/Raw/dirty_cafe_sales.csv')


# 2- PRÉ-VISUALIZAÇÃO

Uma primeira visualização das primeiras linhas do dataset já mostra que há muitos erros de preenchimento.

In [3]:
#Mostrando as 10 primeiras linhas do dataset
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


O dataset possui 10000 linhas e oito colunas.

In [4]:
df.shape

(10000, 8)

Fiz uma verificação preliminar de quantos valores nulos cada coluna possui, e o resultado mostrou que o dataset está inutilizável de tão sujo. São milhares de linhas com valores inválidos. **DEFINIR VALORES INVALIDOS?**

In [5]:
#Total de valores inválidos por coluna
df.isnull().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


In [6]:
#Total geral de colunas com valores inválidos: 6.826
df.isnull().sum().sum()

np.int64(6826)

# 3- FORMATAÇÃO DE CABEÇALHO

Por motivos de clareza, a formatacão do cabeçalho fiz em etapas separadas. Primeiro renomeei o cabeçalho para traduzir o título de cada coluna:

In [7]:
#Renomeando os títulos de cada coluna
df = df.rename(columns= {'Transaction ID': 'id Transacao', 'Items': 'itens', 'Quantity': 'Quantidade', 'Price Per Unit': 'Preco Unitario', 'Total Spent': 'Total Gasto', 'Payment Method': 'Metodo de Pagamento', 'Location': 'Local', 'Transaction Date': 'Data da Compra'} )


A seguir, deixei tudo minusculo:

In [8]:
df.columns = df.columns.str.lower()

Por fim, substituí os espaços por underline, para facilitar o manuseio:

In [9]:
df.columns = df.columns.str.replace(' ', '_')

O resultado ficou assim:

In [10]:
df.head(1)

,id_transacao,item,quantidade,preco_unitario,total_gasto,metodo_de_pagamento,local,data_da_compra
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08


# 4- Tratando valores nulos

Primeiramente, converti em data todos os valores da coluna **data_de_compra** que correspondem ao formato de data:

In [11]:
#Converte a coluna data_da_compra para data onde o conteúdo corresponder a data
df['data_da_compra'] = pd.to_datetime(df['data_da_compra'], errors='coerce')

Agora a coluna **data_de_compra** é do tipo datetime:

In [12]:
#Mostra o tipo de dados da coluna data_de_compra
print(df['data_da_compra'].dtype)

datetime64[ns]


Converti também as colunas **preco_unitario**, **total_gasto** e **quantidade** no tipo numérico, transformando o valores que não fossem numéricos em *NaN* (not a number):

In [13]:
#Converter colunas numéricas
df['preco_unitario'] = pd.to_numeric(df['preco_unitario'], errors='coerce')
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')
df['total_gasto'] = pd.to_numeric(df['total_gasto'], errors='coerce')


Agora a quantidade de valores nulos aumentou ainda mais: de 6.826 para 8.151:

In [14]:
#Total geral de colunas com valores inválidos: 8.151
df.isnull().sum().sum()

np.int64(8151)

Portanto, dividi os valores nulos/inválidos em três categorias, cada uma a ser tratada de uma forma diferente abaixo.







## 4.1- Valores a serem encontrados

In [15]:
#Conta quantas vezes cada valor aparece na coluna total_gasto inclusindo NaN (não-numérico)
df['total_gasto'].value_counts(dropna=False)

,count
total_gasto,
6.0,979
12.0,939
3.0,930
4.0,923
20.0,746
15.0,734
8.0,677
10.0,524
NaN,502


In [16]:
#Conta quantos valores não-numéricos possui a coluna quantidade: 502
df['total_gasto'].isna().sum()

np.int64(502)

### 4.1.1 - Coluna total_gasto

Como **total_gasto = quantidade * preco_unitario**, então onde faltar valor em **total_gasto** ele pode ser descoberto multiplicando **preco_unitario** por **quantidade**:

In [17]:
#Cada linha da coluna total_gasto com valor NaN será preenchido com o valor de quantidade multiplicado pelo valor de preco_unitario naquela linha
df['total_gasto'] = df['total_gasto'].fillna(df['quantidade'] * df['preco_unitario'])

Onde **quantidade** ou **preco_unitário** for um valor inválido, será mantido o *NaN* na coluna **total_gasto**. Neste caso restaram 40 *NaN* na coluna **total_gasto**, o restante das linhas teve o valor correto preenchido.

In [18]:
#Conta quantos NaN tem na coluna total_gasto: 40
df['total_gasto'].isna().sum()

np.int64(40)

In [19]:
#Mostrando nas 10 primeiras linhas como as linhas restantes sempre possuem NaN ou em quantidade ou em preco_unitário, além do total_gasto
df[df['total_gasto'].isna()].head(10)

,id_transacao,item,quantidade,preco_unitario,total_gasto,metodo_de_pagamento,local,data_da_compra
65,TXN_4987129,Sandwich,3.0,NaN,NaN,NaN,In-store,2023-10-20
236,TXN_8562645,Salad,NaN,5.0,NaN,NaN,In-store,2023-05-18
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
641,TXN_2962976,Juice,NaN,3.0,NaN,NaN,NaN,2023-03-17
738,TXN_8696094,Sandwich,NaN,4.0,NaN,NaN,Takeaway,2023-05-14
1674,TXN_9367492,Tea,2.0,NaN,NaN,Cash,In-store,2023-06-19
1761,TXN_3611851,NaN,4.0,NaN,NaN,Credit Card,NaN,2023-02-09
2229,TXN_8498613,Sandwich,2.0,NaN,NaN,NaN,NaN,2023-11-08
2289,TXN_7524977,UNKNOWN,4.0,NaN,NaN,ERROR,NaN,2023-12-09
2585,TXN_1259340,Tea,3.0,NaN,NaN,Digital Wallet,NaN,2023-02-24


Essas 40 linhas com valor nulo em **total_gasto** e também em **quantidade** ou **preco_unitario** serão excluídas:

In [20]:
#Exclui as linhas com NaN na coluna
df = df.dropna(subset=["total_gasto"]);

In [21]:
#Conta quantos NaN tem na coluna total_gasto: nenhum
df['total_gasto'].isna().sum()

np.int64(0)

### 4.1.2 - Coluna preco_unitario

Como **preco_unitario = total_gasto/quantidade**, então onde faltar valor em **preco_unitario** ele pode ser descoberto dividindo **total_gasto** por **quantidade**:

In [22]:
#Cada linha da coluna preco_unitario com valor NaN será preenchido com o valor de quantidade multiplicado pelo valor de total_gasto naquela linha
df['preco_unitario'] = df['preco_unitario'].fillna(df['total_gasto'] / df['quantidade'])

In [23]:
#Mostra as 10 primeiras linhas com NaN na coluna preco_unitario
df[df['preco_unitario'].isna()].head(10)

,id_transacao,item,quantidade,preco_unitario,total_gasto,metodo_de_pagamento,local,data_da_compra
629,TXN_9289174,Cake,NaN,NaN,12.0,Digital Wallet,In-store,2023-12-30
912,TXN_1575608,Sandwich,NaN,NaN,20.0,ERROR,Takeaway,2023-01-05
1008,TXN_7225428,Tea,NaN,NaN,3.0,Credit Card,Takeaway,2023-03-07
1436,TXN_7590801,Tea,NaN,NaN,6.0,Cash,Takeaway,NaT
1482,TXN_3593060,Smoothie,NaN,NaN,16.0,Cash,NaN,2023-03-05
2330,TXN_3849488,Salad,NaN,NaN,5.0,NaN,In-store,2023-03-01
3635,TXN_6177081,Cookie,NaN,NaN,1.0,Cash,ERROR,2023-07-26
3779,TXN_7376255,UNKNOWN,NaN,NaN,25.0,NaN,In-store,2023-05-27
5639,TXN_6206792,Tea,NaN,NaN,6.0,Credit Card,NaN,2023-10-13
5845,TXN_8388462,Smoothie,NaN,NaN,8.0,UNKNOWN,Takeaway,2023-08-19


Restaram 18 linhas nulas, que representam aquelas que não possuem valor de **preco_unitario** e **quantidade** para calcular o **total_gasto**.

In [24]:
#Conta quantos NaN tem na coluna preco_unitario: 18
df['preco_unitario'].isna().sum()

np.int64(18)

Essas 18 linhas com *NaN* na coluna **preco_unitario** foram excluídas:

In [25]:
#Exclui todas as linhas com NaN na coluna preco_unitario
df = df.dropna(subset=['preco_unitario']);

In [26]:
df['preco_unitario'].isna().sum()

np.int64(0)

### 4.1.3 - Coluna quantidade

Como **quantidade = total_gasto/preco_unitario**, então onde faltar valor em **quantidade** ele pode ser descoberto dividindo **total_gasto** por **preco_unitario**.

In [27]:
#Cada linha da coluna quantidade com valor NaN será preenchido com o valor de total_gasto dividido pelo valor de preco_unitario naquela linha
df['quantidade'] = df['quantidade'].fillna(df['total_gasto'] / df['preco_unitario'])

In [28]:
df['quantidade'].isna().sum()

np.int64(0)

Não restou qualquer linha nulas.

Com isso temos as colunas **quantidade**, **preco_unitario** e **total_gasto** sem valores nulos. O total de linhas do dataset caiu de 10.000 para 9.942 — uma perda dentro do aceitável:

In [29]:
df.shape

(9942, 8)

## 4.2- Valores a serem substituídos

Como as colunas **local** e **metodo_de_pagamento** são secundárias, os valores inválidos serão mantidos. Valores como *ERROR* e *UNKNOWN* serão substituídos por *DESCONHECIDO* para facilitar a uma futura análise de dados em Power BI.

### 4.2.1 - Coluna metodo_de_pagamento

In [30]:
#Busca no dataframe valores na coluna "metodo_de_pagamento" diferentes de 'Credit Card', 'Cash', 'Digital Wallet' e os substituem por "DESCONHECIDO"
df.loc[~df['metodo_de_pagamento'].isin(['Credit Card', 'Cash', 'Digital Wallet']
), 'metodo_de_pagamento'] = 'DESCONHECIDO'

Agora na coluna **metodo_de_pagamento** agora há apenas quatro valores: *Credit* *Card*, *Cash*, *Digital Wallet* e *DESCONHECIDO*.


In [31]:
#Conta quantos valores contém a coluna metodo_de_pagamento, incluindo NaN
df['metodo_de_pagamento'].value_counts(dropna=False)

,count
metodo_de_pagamento,
DESCONHECIDO,3158
Digital Wallet,2280
Credit Card,2260
Cash,2244


### 4.2.2 - Coluna local

Já na coluna **local** há apenas os valores: *takeaway*, *in-store*, *NaN*, *ERROR* e *UNKNOWN*. Esses três últimos serão substituídos por DESCONHECIDO

In [32]:
#Conta quantos valores contém a coluna local, incluindo NaN
df['local'].value_counts(dropna=False)

,count
local,
NaN,3249
Takeaway,3004
In-store,2998
ERROR,355
UNKNOWN,336


In [33]:
#Substituindo ERROR e UNKNOWN por DESCONHECIDO na coluna local
df.loc[df['local'].isin(['ERROR', 'UNKNOWN']), 'local'] = 'DESCONHECIDO'

In [34]:
#Substituindo NaN por DESCONHECIDO na coluna local
df['local'] = df['local'].fillna('DESCONHECIDO')

In [35]:
#Conta quantos valores contém a coluna local, incluindo NaN
df['local'].value_counts(dropna=False)

,count
local,
DESCONHECIDO,3940
Takeaway,3004
In-store,2998


Como pode ser visto acima, agora os valores *ERROR* e *UNKNOWN* por *DESCONHECIDO*.

### 4.2.3 - Coluna data_da_compra

In [45]:
#Transformando as datas da coluna data_de_compra em dados do tipo data. Onde o valor não for uma data será considerado valor nulo.
df['data_da_compra'] = pd.to_datetime(df['data_da_compra'], errors='coerce')

## 4.3- Linhas a serem excluídas

Como a coluna **item** é essencial (sem os itens não há venda), nela as linhas com valores inválidos serão excluídas. Lembrando que para não afetar as estatísticas de um dataset de 10 mil linhas o ideal é descartar entre 1 a 3% de linhas, ou seja, entre 150 e 300 linhas.

In [38]:
#Mostra as 20 primeiras linhas
df.head(20)

,id_transacao,item,quantidade,preco_unitario,total_gasto,metodo_de_pagamento,local,data_da_compra
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,DESCONHECIDO,DESCONHECIDO,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,DESCONHECIDO,2023-03-31
6,TXN_4433211,UNKNOWN,3.0,3.0,9.0,DESCONHECIDO,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,DESCONHECIDO,2023-10-28
8,TXN_4717867,NaN,5.0,3.0,15.0,DESCONHECIDO,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5.0,4.0,20.0,DESCONHECIDO,In-store,2023-12-31


In [39]:
#Conta quantos valores contém a coluna item, incluindo NaN
df['item'].value_counts(dropna=False)

,count
item,
Juice,1167
Coffee,1158
Salad,1144
Cake,1134
Sandwich,1122
Smoothie,1089
Cookie,1084
Tea,1081
UNKNOWN,342


In [40]:
#Substituindo ERROR e UNKNOWN por DESCONHECIDO na coluna item
df.loc[df['item'].isin(['ERROR', 'UNKNOWN']), 'item'] = 'DESCONHECIDO'

In [41]:
#Substituindo NaN por DESCONHECIDO na coluna item
df['item'] = df['item'].fillna('DESCONHECIDO')

Se a quantidade de linhas com **item** *DESCONHECIDO* fosse abaixo de 200, eu poderia excluir. Como está em 457, decidi manter para que uma futura análise de dados em Power BI seja considerado o que fazer com essas linhas.

In [42]:
#Conta quantos valores contém a coluna data_da_compra, incluindo NaN
df['data_da_compra'].value_counts(dropna=False)

,count
data_da_compra,
NaT,457
2023-06-16,40
2023-02-06,40
2023-03-13,39
2023-07-24,39
...,...
2023-11-24,15
2023-07-30,15
2023-07-22,14


# 5- Exportação

In [43]:
# Exporta o DataFrame para o formato Parquet
df.to_parquet('cafe_vendas_limpo.parquet', index=False)

In [44]:
from google.colab import files
files.download('cafe_vendas_limpo.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>